# Fact Data Processing Full Load

### Imports

In [0]:
from pyspark.sql.functions import *

### Schema setup for Bronze, Silver and Gold layers

In [0]:
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'

### Defining widgets for runtime configuration
#### Syntax: dbutils.widgets.text(internal handle, pre-filled value, display text)
#### /*.csv = 'read every csv file in this folder'

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(catalog)
print(data_source)

base_path = f's3://databricks-data-engineering-project/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed/'

print(base_path)
print(landing_path)
print(processed_path)

fmcg
orders
s3://databricks-data-engineering-project/orders
s3://databricks-data-engineering-project/orders/landing/
s3://databricks-data-engineering-project/orders/processed/


**Archive Processed Files to Landing if required**

files = dbutils.fs.ls(processed_path)

for file_info in files:<br>
&nbsp;&nbsp;&nbsp;&nbsp;dbutils.fs.mv(file_info.path, f'{landing_path}/{file_info.name}', True)

### Read CSV from source

In [0]:
df = (
    spark.read.format('csv')
    .option('header', True)
    .option('inferSchema', True)
    .load(f'{landing_path}/*')
    .withColumn('read_timestamp', current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)

display(df.limit(10))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FOCT62720602,"Tuesday, September 30, 2025",ABC987,25891301,71.0,2026-10-23T12:46:38.202Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891502,125.0,2026-10-23T12:46:38.202Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891403,462.0,2026-10-23T12:46:38.202Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",INVALID,25891601,133.0,2026-10-23T12:46:38.202Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891602,79.0,2026-10-23T12:46:38.202Z,orders_2025_09_30.csv,41446
FOCT62720602,2025/09/30,XYZ123,25891101,472.0,2026-10-23T12:46:38.202Z,orders_2025_09_30.csv,41446
FOCT62720602,2025/09/30,ABC987,25891301,71.0,2026-10-23T12:46:38.202Z,orders_2025_09_30.csv,41446
FOCT62402303,"Tuesday, September 30, 2025",789402,25891303,24.0,2026-10-23T12:46:38.202Z,orders_2025_09_30.csv,41446
FOCT62402303,"Tuesday, September 30, 2025",789402,25891301,26.0,2026-10-23T12:46:38.202Z,orders_2025_09_30.csv,41446
FOCT62402303,"Tuesday, September 30, 2025",789402,25891102,328.0,2026-10-23T12:46:38.202Z,orders_2025_09_30.csv,41446


### Inspect the inferred schema

In [0]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



### Ingesting raw data into Bronze layer

In [0]:
(
    df.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')
)

### Archive Landing Files to Processed

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(file_info.path, f'{processed_path}/{file_info.name}', True)

### Silver Processing

In [0]:
df_bronze = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source};')
df_bronze.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-10-23 12:46:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-10-23 12:46:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-2025|     789320|  25891403|     NULL|2026-10-23 12:46:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891201|    354.0|2026-10-23 12:46:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891501|    249.0|2026-10-23 12:46:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-20

### Drop Rows with Null `order_qty`

In [0]:
df_silver = df_bronze.filter(col("order_qty").isNotNull())

### Sanitize `customer_id` column

In [0]:
df_silver = (
    df_silver.withColumn('customer_id',
    when(col('customer_id').rlike(r'^[0-9]+$'), col('customer_id').cast('string'))
    .otherwise(lit(999999).cast('string')))
)

In [0]:
df_silver.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-10-23 12:46:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-10-23 12:46:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891201|    354.0|2026-10-23 12:46:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891501|    249.0|2026-10-23 12:46:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-2025|     789320|  25891301|     46.0|2026-10-23 12:46:...|orders_2025_07_01...|    20744|
|FJUL33401603|Tuesday, July 01,.

### Sanitize `order_placement_date` values

In [0]:
df_silver = (
    df_silver.withColumn('order_placement_date', 
    regexp_replace(col('order_placement_date'), r'^[A-Za-z]+,\s*', ''))
)

### Parse `order_placement_date` from multiple possible formats

In [0]:
df_silver = (
    df_silver.withColumn('order_placement_date',
    coalesce(
    try_to_date(col('order_placement_date'), 'yyyy/MM/dd'),
    try_to_date(col('order_placement_date'), 'dd/MM/yyyy'),
    try_to_date(col('order_placement_date'), 'yyyy-MM-dd'),
    try_to_date(col('order_placement_date'), 'dd-MM-yyyy'),
    try_to_date(col('order_placement_date'), 'MMMM dd, yyyy')
    ))
)            

### Drop duplicates

In [0]:
df_silver = (
    df_silver.drop_duplicates(subset = 
    ['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'])
)

### Cast `product_id` to string

In [0]:
df_silver = (
    df_silver.withColumn('product_id', col('product_id').cast('string'))
)

### Sanity Check: Earliest and Latest Dates

In [0]:
df_silver.select(
    min('order_placement_date').alias('min_date'),
    max('order_placement_date').alias('max_date')
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-07-01|2025-11-30|
+----------+----------+



#### Enrich the silver dataset by performing an inner join with the `products` table to retrieve the `product_code` for each `product_id`

In [0]:
df_products = spark.table(f'{catalog}.{silver_schema}.products')
df_joined = df_silver.join(df_products.select('product_id', 'product_code'), on = 'product_id', how = 'inner')

display(df_joined.limit(10))

product_id,order_id,order_placement_date,customer_id,order_qty,read_timestamp,file_name,file_size,product_code
25891103,FJUL32202603,2025-07-01,789202,370.0,2026-10-23T12:46:47.169Z,orders_2025_07_01.csv,20744,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268
25891602,FJUL33321602,2025-07-01,789321,74.0,2026-10-23T12:46:47.169Z,orders_2025_07_01.csv,20744,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
25891102,FJUL34303602,2025-07-01,789303,317.0,2026-10-23T12:46:47.169Z,orders_2025_07_01.csv,20744,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
25891402,FJUL34622602,2025-07-01,789622,439.0,2026-10-23T12:46:47.169Z,orders_2025_07_01.csv,20744,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49
25891303,FJUL33402303,2025-07-01,789402,63.0,2026-10-23T12:46:47.169Z,orders_2025_07_01.csv,20744,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f
25891602,FJUL33721603,2025-07-01,999999,148.0,2026-10-23T12:46:47.169Z,orders_2025_07_01.csv,20744,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
25891602,FJUL32420602,2025-07-01,789420,76.0,2026-10-23T12:46:47.169Z,orders_2025_07_01.csv,20744,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
25891602,FJUL33201602,2025-07-01,789201,52.0,2026-10-23T12:46:47.169Z,orders_2025_07_01.csv,20744,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
25891101,FJUL33403501,2025-07-01,789403,379.0,2026-10-23T12:46:47.169Z,orders_2025_07_01.csv,20744,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
25891203,FJUL33421203,2025-07-01,789421,99.0,2026-10-23T12:46:47.169Z,orders_2025_07_01.csv,20744,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2


### Write Transformed Data to Silver

In [0]:
(
    df_joined.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .option('overwriteSchema', True)
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Gold Processing - Select final columns

In [0]:
df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source}')
df_gold = df_silver.select('order_id', 'order_placement_date', 'customer_id', 'product_id', 'product_code', 'order_qty')

### Writing staging dimension to gold schema

In [0]:
(
    df_gold.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{gold_schema}.staging_fact_{data_source}')
)

### Aggregate Monthly Order Quantities by Product and Customer

In [0]:
df_monthly = (
    df_gold
    .withColumn('month_start', trunc(col('order_placement_date'), 'MM'))
    .groupBy(col('month_start'), col('product_code'), col('customer_id'))
    .agg(sum(col('order_qty')).alias('sold_quantity'))
    .withColumnRenamed('month_start', 'date')
    .withColumnRenamed('customer_id', 'customer_code')
)

display(df_monthly.limit(10))

date,product_code,customer_code,sold_quantity
2025-07-01,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,789202,2944.0
2025-07-01,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,789321,1742.0
2025-07-01,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,789303,4929.0
2025-07-01,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,789622,3810.0
2025-07-01,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,789402,960.0
2025-07-01,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,999999,6938.0
2025-07-01,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,789420,1565.0
2025-07-01,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,789201,1691.0
2025-07-01,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,789403,5872.0
2025-07-01,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,789421,3318.0


### Stage `df_monthly` as TempView for MERGE

In [0]:
df_monthly.createOrReplaceTempView('monthly_src')

### Merge TempView with parent

In [0]:
display(spark.sql(f"""
MERGE INTO {catalog}.{gold_schema}.fact_{data_source} AS target
USING monthly_src AS source
ON
    target.date = source.date
AND
    target.product_code = source.product_code
AND
    target.customer_code = source.customer_code
-- match on full monthly grain (date + product + customer) to prevent multi-month collisions

WHEN MATCHED THEN
UPDATE
    SET
        sold_quantity = source.sold_quantity

WHEN NOT MATCHED THEN
INSERT (
    date,
    product_code,
    customer_code,
    sold_quantity
)
VALUES(
    source.date,
    source.product_code,
    source.customer_code,
    source.sold_quantity
);
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3060,0,0,3060
